# SMX Quickstart: Binary Classification on a Synthetic Spectral Dataset

This notebook presents a complete and reproducible SMX workflow for binary classification on synthetic spectral data. The pipeline covers data generation, automatic zone detection, model training, and a suite of visualization tools for interpreting the classifier's decision logic.

**Workflow steps:**
- Generate a synthetic spectral dataset with two classes differing in peak amplitudes and widths
- Automatically detect spectral zones via data-driven peak detection (no a priori zone definitions)
- Split data into calibration and test sets, applying mean-centering using calibration statistics only
- Train an RBF-SVM classifier and derive probabilistic outputs for SMX interpretation
- Fit the SMX explanation pipeline on calibration data using bagging and perturbation-based scoring
- Rank predicates by Local Reaching Centrality (LRC) to identify the most influential decision rules
- Reconstruct top-ranked predicate thresholds back into the original spectral domain
- Visualize zone discriminative power, threshold boundaries, LRC rankings, and faithfulness diagnostics

In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
import smx
from smx import SMX, generate_synthetic_spectral_data, plot_zone_ranking_over_spectrum
from smx.zones import building_spectral_zones
from smx.graph.interpretation import reconstruct_threshold_to_spectrum
import plotly.graph_objects as go
SEED = 42
from IPython.display import HTML, IFrame

## Synthetic Spectral Dataset

The synthetic data were generated by modeling each spectrum as a superposition of Gaussian peaks plus additive noise. Formally, each synthetic spectrum was described by

$S(x) = \sum_{i} A_i \exp\!\left(-\frac{(x - c_i)^2}{2\sigma_i^2}\right) + \varepsilon(x)$,

where $c_i$, $A_i$ and $\sigma_i$ are, respectively, the center, amplitude and width (standard deviation) of the $i$-th Gaussian peak, and $\varepsilon(x)\sim\mathcal{N}(0,\,\sigma_\varepsilon^2)$ is a white-noise baseline. To introduce realistic sample-to-sample variability, both $A_i$ and $\sigma_i$ were drawn independently for each sample from normal distributions: $A_i \sim \mathcal{N}(\bar{A},\,s_A^2)$ and $\sigma_i \sim \mathcal{N}(\bar{\sigma},\,s_\sigma^2)$.

Two classes with the following peak configurations were defined:

- **Class A**: peaks centered at 150, 300, and 500 (channel units)
- **Class B**: peaks centered at 150, 300, and 500 with distinct amplitudes and widths

The spectral axis is sampled at **300 points** from **1 to 600**, enabling controlled class separation while preserving realistic overlap.

In [2]:
CLASSES_CONFIG = [
    {
        "name": "A",
        "n_samples": 116,
        "peaks": [
            {'center' : 150, 'amplitude_mean' : 2.5, 'amplitude_std' : 0.5, 'width_mean' : 15.0, 'width_std' : 2.0},
            {'center' : 300, 'amplitude_mean' : 1.8, 'amplitude_std' : 0.3, 'width_mean' : 15.0, 'width_std' : 2.0},
            {'center' : 500, 'amplitude_mean' : 0.5, 'amplitude_std' : 0.3, 'width_mean' : 15.0, 'width_std' : 2.0},
        ],
        "noise_std": 0.08,
    },
    {
        "name": "B",
        "n_samples": 126,
        "peaks": [
            {'center' : 150, 'amplitude_mean' : 3.3, 'amplitude_std' : 0.3, 'width_mean' : 17.0, 'width_std' : 2.0},
            {'center' : 300, 'amplitude_mean' : 0.8, 'amplitude_std' : 0.3, 'width_mean' : 14.0, 'width_std' : 1.5},
            {'center' : 500, 'amplitude_mean' : 0.45, 'amplitude_std' : 0.3, 'width_mean' : 15.0, 'width_std' : 2.0},
        ],
        "noise_std": 0.1,
    },
]

df = generate_synthetic_spectral_data(
    classes_config=CLASSES_CONFIG,
    n_points=300,
    x_min=1,
    x_max=600,
    seed=SEED,
)

X = df.drop(columns=["Class"])
y = df["Class"]

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{y.value_counts().to_string()}")
df.head()

Dataset shape: (242, 301)
Class distribution:
Class
B    126
A    116


,Class,1.0,3.0033444816053514,5.006688963210703,7.010033444816054,9.013377926421406,11.016722408026757,13.020066889632108,15.02341137123746,17.02675585284281,...,581.9698996655519,583.9732441471573,585.9765886287626,587.9799331103679,589.9832775919733,591.9866220735787,593.989966555184,595.9933110367894,597.9966555183947,600.0
0,A,0.039737,-0.011061,0.051815,0.121842,-0.018732,-0.018731,0.126337,0.061395,-0.037558,...,-0.016650,-0.039440,-0.047149,0.067968,0.028561,-0.055433,0.071968,0.024584,0.065029,0.050370
1,A,0.102213,-0.047326,0.043768,-0.016175,-0.017414,0.087902,0.066033,0.065081,0.104438,...,-0.021590,-0.078301,-0.035543,0.030184,0.060559,-0.073773,0.069568,0.108451,0.033075,0.150144
2,A,0.022397,-0.090039,0.195660,0.010338,0.008752,0.058061,0.038481,0.017911,-0.063238,...,0.002300,0.102276,0.015288,0.003715,-0.108788,0.059700,0.051639,0.173060,-0.024622,0.017532
3,A,-0.035715,0.015527,0.085891,-0.082121,0.010638,-0.056010,0.095604,-0.121855,-0.044714,...,-0.112207,0.139967,-0.099509,-0.055432,-0.057473,0.071594,-0.023596,0.099819,-0.053879,0.022320
4,A,-0.014823,-0.010386,0.003505,-0.011760,0.077110,0.176842,-0.044599,-0.109584,-0.007063,...,-0.006614,-0.009740,0.121076,0.050465,-0.081935,0.148327,0.097683,0.046568,-0.018119,-0.076755


Since spectral zones are not known a priori in this example, the `building_spectral_zones` function is applied to automatically identify zones based on peak detection criteria. The function uses the mean spectrum across all samples (obtained by passing the full dataset `X`) to detect local maxima and minima, which define the zone boundaries. Two key parameters control the detection sensitivity:

- **`min_window_length`**: Minimum length of spectral windows considered for minima extraction, which also controls the minimum width of the resulting zones.
- **`prominence`**: Minimum difference in intensity between a peak and its surrounding baseline for it to be considered a valid peak. Higher values make detection more robust to noise but may miss subtle features.
- Optionally, Savitzky-Golay smoothing can be applied before peak detection (`svg_smooth=True`) to further improve robustness in noisy spectra.

In [5]:
spectral_cuts = building_spectral_zones(
    spectrum=X, # when the entire dataset is passed, the mean spectrum across samples is used for zone detection; alternatively, a single representative spectrum can be passed (e.g., the spectrum of a specific sample or the mean spectrum of a specific class)
    min_window_length=1, # the minimum length of the spectral windows considered for minima extraction (and thus controls the minimum width of the resulting zones)
    prominence=0.5, # the minimum difference in intensity between a peak and its surrounding baseline for it to be considered a valid peak, being a key parameter for extracting peaks robustly in the presence of noise
    svg_smooth=False, # whether to apply Savitzky-Golay smoothing to the spectrum before peak detection (which can help improve peak detection in noisy spectra); if True, the svg_polyorder and svg_window_length parameters can be set to control the degree of smoothing
    svg_polyorder=3, # unused when svg_smooth=False, but can be set to control the degree of smoothing if svg_smooth=True
    svg_window_length=11, # unused when svg_smooth=False, but can be set to control the window length of smoothing if svg_smooth=True
    _show_minima=False, # whether to show the detected minima in the plot (which define the zone boundaries)
    ploting=True # whether to plot the spectrum with detected peaks and zone boundaries (which can help visually assess the quality of zone detection and adjust parameters accordingly
)
print('Spectral cuts (label, start, end):')
for label, start, end in spectral_cuts:
    print(f"{label}: {start} - {end}")

Spectral cuts (label, start, end):
background1: 1.0 - 93.15384615384616
zone1: 93.15384615384616 - 203.3377926421405
background2: 203.3377926421405 - 259.4314381270903
zone2: 259.4314381270903 - 341.56856187290975
background3: 341.56856187290975 - 467.77926421404686
zone3: 467.77926421404686 - 535.8929765886288
background4: 535.8929765886288 - 600.0


The following cell overlays all spectra from both classes, colored by class label, with vertical dashed lines marking the zone boundaries detected above. Zone labels are annotations centered within each zone. This visualization confirms that the synthetic design produces distinguishable spectral profiles: Class A (red) and Class B (blue) show systematic differences in peak intensity at the three discriminative regions (near 150, 300, and 500), while the baseline regions exhibit overlapping behavior. The detected zones correctly segment informative peaks from background intervals.

In [6]:
# Plotting the entire synthetic dataset with spectral cuts

data_complete = df.copy() # using the synthetic dataset generated above

class_col = "Class" if "Class" in data_complete.columns else data_complete.columns[0]
spec_cols = [c for c in data_complete.columns if c != class_col]
x_vars = np.array(spec_cols, dtype=float)

df_A = data_complete[data_complete[class_col] == "A"][spec_cols]
df_B = data_complete[data_complete[class_col] == "B"][spec_cols]

color_A = "#FA0F0F"
color_B = "#1365FC"

fig = go.Figure()

for i, (_, row) in enumerate(df_A.iterrows()):
    fig.add_trace(go.Scatter(
        x=x_vars,
        y=row.values,
        mode="lines",
        name="Class A",
        line=dict(color=color_A, width=1.0),
        legendgroup="A",
        showlegend=(i == 0),
    ))

for i, (_, row) in enumerate(df_B.iterrows()):
    fig.add_trace(go.Scatter(
        x=x_vars,
        y=row.values,
        mode="lines",
        name="Class B",
        line=dict(color=color_B, width=1.0),
        legendgroup="B",
        showlegend=(i == 0),
    ))

boundaries = sorted({cut[1] for cut in spectral_cuts} | {cut[2] for cut in spectral_cuts})
for b in boundaries:
    fig.add_vline(x=b, line=dict(color="gray", width=5, dash="dot"))

y_max = data_complete[spec_cols].values.max()
label_y = y_max * 1.02

for name, x_start, x_end in spectral_cuts:
    fig.add_annotation(
        x=(x_start + x_end) / 2,
        y=label_y,
        text=name,
        showarrow=False,
        textangle=-90,
        font=dict(size=20, color="black"),
        xanchor="center",
        yanchor="bottom",
    )

tick_max = int(np.ceil(x_vars.max() / 50.0) * 50)
ticks = np.arange(0, tick_max + 1, 50)

fig.update_layout(
    font=dict(size=19),
    xaxis_title="Variables (a.u.)",
    xaxis=dict(tickvals=ticks, ticktext=[str(v) for v in ticks]),
    yaxis_title="Intensity (a.u.)",
    yaxis=dict(range=[data_complete[spec_cols].values.min(), y_max * 1.35]),
    legend=dict(
        orientation="h",
        x=0.86,
        y=-0.3,
        xanchor="center",
        yanchor="bottom",
        font=dict(size=20),
    ),
    template="simple_white",
    #width=1250,
    #height=350,
)
fig.show()

The overlaid spectra and zone boundaries above provide a qualitative overview of the synthetic design. The three informative zones (around 150, 300, and 500) contain the class-discriminative peaks, while the remaining zones represent background regions with no class separation. This visual check confirms that the data-driven zone detection successfully isolated the spectrally active regions from baseline intervals. The peak near 150 shows the largest class separation, followed by the peaks at 300 and 500, establishing a ground-truth hierarchy for evaluating whether SMX recovers the same ordering.

## Calibration-Test Partition and Mean-Centering Preprocessing

The dataset is stratified to preserve class proportions across subsets. Mean-centering is then performed using calibration-set statistics only, preventing information leakage from the test set.

In [7]:
X_cal, X_test, y_cal, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_cal = X_cal.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_cal = y_cal.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Mean-centre (fit on calibration set only)
X_mean = X_cal.mean()
X_cal_prep = X_cal - X_mean
X_test_prep = X_test - X_mean

print(f"Calibration set: {X_cal.shape}  |  Test set: {X_test.shape}")
print(f"Cal class distribution: {y_cal.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}")

Calibration set: (169, 300)  |  Test set: (73, 300)
Cal class distribution: {'B': 88, 'A': 81}
Test class distribution: {'B': 38, 'A': 35}


The calibration-test split is stratified to preserve class balance. Mean-centering is estimated exclusively on calibration observations and then applied to both subsets, which ensures methodological rigor under a strict out-of-sample protocol.

## SVM Classifier

An RBF-SVM is trained on preprocessed calibration data. Its probabilistic output is subsequently used as the continuous response variable required by the SMX perturbation analysis.

In [8]:
svm = SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED)
svm.fit(X_cal_prep, y_cal)

acc = (svm.predict(X_test_prep) == y_test).mean()
print(f"SVM test accuracy: {acc:.2%}")

# Continuous output used by SMX (probability of class A)
class_order = list(svm.classes_)
class_a_idx = class_order.index("A")
y_pred_cal = pd.Series(svm.predict_proba(X_cal_prep)[:, class_a_idx])
print(f"y_pred_cal range: [{y_pred_cal.min():.3f}, {y_pred_cal.max():.3f}]")

SVM test accuracy: 94.52%
y_pred_cal range: [0.000, 1.000]


Model performance on the held-out test set quantifies predictive validity, whereas calibration-set class probabilities provide the continuous model response required by perturbation-based explainability in SMX.

## SMX Pipeline

SMX is configured using the spectral partition automatically derived from the `building_spectral_zones` function. This partition supports the interpretation of class-related discriminative regions while preserving non-informative baseline intervals.

In [9]:
smx = SMX(
    spectral_cuts=spectral_cuts, # defined in the plotting cell above
    quantiles=[0.2, 0.4, 0.6, 0.8], # quantiles thresholding the spectrum for deriving the prediates
    n_repetitions=4, # number of repetitions of the entire SMX pipeline 
    n_bags=20, # number of bags for the bagging procedure
    n_samples_fraction=0.8, # fraction of samples to use in each bag 
    metric="perturbation", # scoring the predicates impact by perturbing the zones via masking with median value
    estimator=svm, # the fitted SVM model is passed to SMX for interpreting its predictions
    perturbation_metric="probability_shift", # the impact of perturbation is quantified by the shift in predicted probability
)

print("Fitting SMX pipeline…")
smx.fit(X_cal_prep, y_pred_cal, X_cal_natural=X_cal)
print("Done.")

Fitting SMX pipeline…
Bag_1 | samples: yes | predicates: no (all) | valid: 44 | discarded: 12
Bag_2 | samples: yes | predicates: no (all) | valid: 43 | discarded: 13
Bag_3 | samples: yes | predicates: no (all) | valid: 43 | discarded: 13
Bag_4 | samples: yes | predicates: no (all) | valid: 43 | discarded: 13
Bag_5 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_6 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_7 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_8 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_9 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_10 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_11 | samples: yes | predicates: no (all) | valid: 43 | discarded: 13
Bag_12 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_13 | samples: yes | predicates: no (all) | valid: 42 | discarded: 14
Bag_14 | samples: yes | predicates: no

At this stage, SMX is fitted on calibration spectra and corresponding model outputs. The selected hyperparameters control bagging stability, sampling fractions, and perturbation behavior, thereby defining the robustness and granularity of the explanatory graph.

After fitting, the SMX object exposes several key attributes for interpretation:
- `lrc_natural_`: DataFrame with LRC scores for every predicate, in original spectral units (zone labels, operators, thresholds).
- `lrc_pca_`: Same ranking but expressed in PCA-transformed coordinates.
- `zones_natural_`: The mean spectrum per zone used for threshold reconstruction.
- `pca_info_natural_`: PCA loadings and model metadata required for inverse transformations.
- `predicate_graph_`: The underlying graph structure connecting predicates to zones.

These attributes feed directly into the visualization functions demonstrated below.

## Results: Top Predicates by Local Reaching Centrality

Predicates are ranked by Local Reaching Centrality (LRC), highlighting the most influential decision rules in the SMX explanatory graph.

In [10]:
top = (
    smx.lrc_natural_[smx.lrc_natural_["Zone"].notna()]
    .sort_values("Local_Reaching_Centrality", ascending=False)
    [["Node", "Zone", "Operator", "Threshold_Natural", "Local_Reaching_Centrality"]]
    .reset_index(drop=True)
)

print("Top predicates by Local Reaching Centrality:")
top.head(30)

Top predicates by Local Reaching Centrality:


,Node,Zone,Operator,Threshold_Natural,Local_Reaching_Centrality
0,zone2 > 0.88,zone2,>,0.891528,8.520180
1,zone2 > -1.05,zone2,>,-1.061298,7.964055
2,zone2 <= 1.77,zone2,<=,1.757621,7.407452
3,zone2 <= -1.05,zone2,<=,-1.061298,7.175069
4,zone2 > -1.98,zone2,>,-1.941480,6.946076
5,zone2 <= 0.88,zone2,<=,0.891528,6.358727
6,zone2 <= -1.98,zone2,<=,-1.941480,6.146007
7,zone1 > 1.22,zone1,>,1.231347,4.459842
8,zone1 > -0.61,zone1,>,-0.608105,3.181629
9,zone1 > -2.39,zone1,>,-2.315951,2.731277


The ranked table above reveals the structural importance of each predicate in the SMX explanatory graph. Predicates with higher LRC values propagate influence across more downstream logical relations, indicating decision boundaries that strongly shape the model's behavior. The hierarchy aligns with the expected discriminative power derived from the synthetic design: predicates in the zone around 150 (highest class separation) dominate, followed by the zones near 300 and 500. The top predicates serve as the basis for the threshold reconstruction and zone-ranking visualizations in the sections that follow.

## SMX Plotting Gallery

With predicates ranked and the SMX model fitted, we now demonstrate the visualization suite. Each function consumes the `smx` object's output attributes and returns either a Plotly figure or a diagnostic DataFrame. A consistent visual theme (`SMXTheme`) is defined first to unify fonts, colorscales, class colors, and opacity settings across all plots.

In [11]:
from smx import SMXTheme

my_theme = SMXTheme(
    template="simple_white", # plotly template to use as base for all SMX plots, which can be customized via the other parameters below (e.g., colorscale, font_family, etc.)
    font_family="Georgia, serif", # font family for all text elements in the plots, which can be set to any valid CSS font-family string (e.g., "Arial, sans-serif" or "Times New Roman, serif")
    font_size=15, # base font size for all text elements in the plots, which can be adjusted to improve readability and aesthetics of the plots
    colorscale="Blues",
    class_colors={"A": "#0099ff", "B": "#02005C"},
    threshold_color="#dd0707",
    zone_opacity=0.80,
)

### `plot_zone_ranking_over_spectrum`

Colored bands encode the LRC score of each zone, overlaid on the mean calibration spectrum. A vertical colorbar provides the mapping between band color and LRC magnitude. Class-specific spectra (optional, shown here with dashed lines) can be layered to highlight where each class deviates most. Zones with the highest LRC appear most saturated, immediately revealing the most discriminative spectral regions.

In [12]:
from smx import plot_zone_ranking_over_spectrum

plot_zone_ranking_over_spectrum(
    zone_ranking_df=smx.lrc_natural_,
    spectral_cuts=spectral_cuts,
    reference_spectrum=smx.zones_natural_,
    output_path=None,
    title="Zone ranking over spectrum",
    spectrum_name="Mean calibration spectrum",
    class_spectra={"A": X_cal[y_cal == "A"], "B": X_cal[y_cal == "B"]},
    class_colors={"A": "#e41a1c", "B": "#377eb8"},
    theme=my_theme,
    return_df=True,
)

,zone,score,rank
0,zone2,8.520180,1
1,zone1,4.459842,2
2,zone3,1.003259,3
3,background2,0.003328,4
4,background3,0.002297,5
5,background4,0.002281,6
6,background1,0.002277,7


### `plot_threshold_spectrum`

For the top-ranked predicate, the threshold computed in PCA space is inversely transformed and overlaid on the original calibration spectra, grouped by class. A vertical dashed line marks the reconstructed boundary within the zone. This plot directly connects abstract graph-based predicates to physically interpretable spectral profiles, showing how the decision rule partitions the two classes along the spectral axis.

In [ ]:
from smx import plot_threshold_spectrum

plot_threshold_spectrum(
    lrc_natural_df=smx.lrc_natural_,
    row_index=0, # index of the row in lrc_natural_df corresponding to the predicate to plot (e.g., top-ranked predicate is at index 0, second-ranked predicate is at index 1, etc.)
    spectral_zones_original=smx.zones_natural_,
    pca_info_dict_original=smx.pca_info_natural_,
    y_labels=y_cal,
    output_path=None,
    class_colors={"A": "#e41a1c", "B": "#377eb8"},
    theme=my_theme,
    return_df=True,
)

259.4314381270903     0.050218
261.4347826086957     0.069203
263.438127090301      0.077434
265.4414715719064     0.112056
267.4448160535117     0.146459
269.4481605351171     0.179847
271.4515050167224     0.248517
273.4548494983278     0.320026
275.4581939799331     0.378655
277.4615384615385     0.460925
279.46488294314383    0.577607
281.4682274247492     0.676512
283.47157190635454    0.797760
285.4749163879599     0.908430
287.47826086956525    1.029792
289.48160535117063    1.144127
291.48494983277595    1.250727
293.4882943143813     1.336927
295.49163879598666    1.394081
297.494983277592      1.447378
299.49832775919737    1.482033
301.5016722408027     1.471721
303.5050167224081     1.432297
305.5083612040134     1.363134
307.5117056856188     1.299011
309.5150501672241     1.209262
311.5183946488295     1.092461
313.5217391304348     0.971872
315.5250836120402     0.840072
317.5284280936455     0.737085
319.5317725752509     0.619477
321.5351170568562     0.527680
323.5384

### `plot_lrc_bar`

Bar chart showing the maximum LRC score achieved within each spectral zone, sorted descending. Bar color maps to the same colorscale used in the zone-ranking plot above, enabling direct visual correspondence between the two figures. This view is useful for quickly identifying which zones dominate the explanation regardless of how many predicates they contain.

In [18]:
from smx import plot_lrc_bar

plot_lrc_bar(
    zone_ranking_df=smx.lrc_natural_,
    output_path=None,
    title="Max LRC importance score by spectral zone",
    width=800,
    height=500,
    theme=my_theme,
    return_df=True
)

,zone,score,rank,pct
6,background1,0.002277,7,0.016273
5,background4,0.002281,6,0.016300
4,background3,0.002297,5,0.016412
3,background2,0.003328,4,0.023781
2,zone3,1.003259,3,7.169483
1,zone1,4.459842,2,31.870897
0,zone2,8.520180,1,60.886854


### `plot_predicate_heatmap`

Heatmap with zones as rows (sorted by highest LRC at the top) and predicates as columns grouped by operator (`≤` then `>`). Color encodes LRC score; grey cells indicate a predicate does not appear in that zone. This matrix provides a compact overview of where each zone's explanatory power comes from — which operators and thresholds drive its influence in the explanatory graph.

In [19]:
from smx import plot_predicate_heatmap

plot_predicate_heatmap(
    lrc_natural_df=smx.lrc_natural_,
    output_path=None,
    theme=my_theme,
    return_df=True
)

predicate_label,≤ T1,≤ T2,≤ T3,≤ T4,> T1,> T2,> T3,> T4
Zone,,,,,,,,
background1,0.001854,0.001761,0.002083,0.002277,0.001958,0.002151,0.001907,0.000599
background4,0.002281,0.001966,0.002138,NaN,0.002201,0.002031,0.001959,0.000976
background3,0.001332,0.002101,0.001849,NaN,0.002297,0.002137,0.001933,0.001569
background2,0.003328,0.002483,0.002668,NaN,0.001962,0.001334,0.000897,0.000686
zone3,0.468851,0.338926,0.135995,NaN,0.618855,0.771387,1.003259,0.631635
zone1,2.087620,1.348602,2.599860,NaN,2.731277,3.181629,4.459842,NaN
zone2,6.146007,7.175069,6.358727,7.407452,6.946076,7.964055,8.520180,NaN


### `plot_zone_scores`

Split-violin plot of PC1 scores per spectral zone, with violins mirrored by class. For binary classification the violins are displayed back-to-back, making it easy to spot zones where class distributions are well separated in the compressed (PCA) space. Zone-score distributions complement the LRC-based ranking by revealing the raw score separation that drives the explanatory importance.

In [20]:
from smx import extract_spectral_zones, ZoneAggregator, plot_zone_scores

spectral_zones_cal = extract_spectral_zones(X_cal_prep, spectral_cuts)
agg = ZoneAggregator(method='pca')
agg.fit(spectral_zones_cal)

plot_zone_scores(
    zones=spectral_zones_cal,
    y_labels=y_cal,
    output_path=None,
    class_colors={"A": "#e41a1c", "B": "#377eb8"},
    theme=my_theme,
    return_df=True,
)

,background1,zone1,background2,zone2,background3,zone3,background4
0,-0.177495,2.426883,0.188465,-3.306699,0.221480,-0.426656,0.060913
1,0.011514,-5.186826,0.191236,-0.280396,0.118595,-0.511703,-0.046192
2,-0.289420,3.080917,-0.013089,-1.157153,0.064683,-0.187453,-0.023166
3,0.064250,-0.608105,0.001028,1.034693,0.127372,0.948168,-0.055714
4,-0.016702,-0.593025,-0.061998,0.370754,0.061918,-1.581499,0.071932
...,...,...,...,...,...,...,...
164,0.065022,4.106965,0.135494,-1.247968,-0.321296,1.470485,0.095728
165,-0.228033,1.988510,0.020788,-0.715425,0.203445,0.181301,-0.222189
166,-0.130486,1.824745,-0.188160,-2.129045,-0.497763,-1.050733,0.194624
167,0.052839,-2.777562,-0.052655,0.382135,-0.156082,-1.319913,-0.005187


### `plot_faithfulness_curve`

Progressive masking faithfulness diagnostic: zones are masked in order of decreasing LRC and the accumulated prediction shift on the test set is plotted (trapezoidal AUC shaded). The curve's AUC, normalized AUC, categorical level, and percentile against a random baseline are annotated on the figure. A curve that rises steeply and saturates early indicates that top-ranked zones carry almost all discriminative information — a hallmark of a faithful explanation.

The output of `evaluate_faithfulness` includes:

- **`auc`** — trapezoidal AUC of the masking curve. Raw AUC values are bounded by the total number of zones and the model's baseline accuracy; they are **normalised to the [0, 1] interval** by dividing by the maximum achievable AUC (i.e. the AUC of a perfectly ordered ranking that masks the least-informative zones first).
- **`level`** — a categorical quality label assigned as follows:

  | Level | Condition |
  |-------|-----------|
  | *very high* | `null_percentile ≥ 95` |
  | *high* | `null_percentile ≥ 90` |
  | *moderate* | `null_percentile ≥ 75` |
  | *low* | `null_percentile ≥ 50` |
  | *very low* | `null_percentile < 50` |

- **`null_percentile`** — percentile of the true AUC against a **null distribution** built by computing the AUC for a large number of random zone orderings (default: 500 permutations). A percentile close to 100 means the LRC-based ranking is far better than random; a percentile near 50 means the ranking carries no more information than chance.
- **`curve_df`** — a DataFrame with columns `k`, `masked_zone`, `masked_zones`, and `score` describing the curve point at each masking step
- **`plot_path`** — path to a saved interactive Plotly HTML figure (when `output_path` is provided)
- **`null_distribution`** — list of AUC values from the null (random) permutations, useful for diagnostic histograms
- **`k`** — number of top zones at which the maximum drop in prediction score is observed

**Interpretation guide for the masking curve:**

- **Steep early drop** (top-left of curve is high) — the first few zones dominate the prediction, confirming that the LRC ranking captures the core decision boundary
- **Gradual decline** — predictive power is distributed across many zones; the model relies on a broad spectral signature rather than a few sharp features
- **Flat curve** — masking has little effect regardless of zone order, indicating either a weak classifier or a ranking that is misaligned with decision behaviour


In [21]:
from smx import plot_faithfulness_curve

faithfulness = smx.evaluate_faithfulness(
    X_test_prep,
    ranking="unique",
    masking_strategy="zero",
    output_path=None,
)

plot_faithfulness_curve(
    faithfulness_result=faithfulness,
    output_path=None,
    theme=my_theme,
    return_df=True,
)

,k,masked_zone,masked_zones,score
0,1,zone2,"(zone2,)",0.164485
1,2,zone1,"(zone2, zone1)",0.456161
2,3,zone3,"(zone2, zone1, zone3)",0.457341
3,4,background2,"(zone2, zone1, zone3, background2)",0.458559
4,5,background3,"(zone2, zone1, zone3, background2, background3)",0.457533
5,6,background4,"(zone2, zone1, zone3, background2, background3...",0.459536
6,7,background1,"(zone2, zone1, zone3, background2, background3...",0.458587
